# AP Commander — GRPO Training Pipeline

**Environment:** `https://pathikreet-ap-clerk-env.hf.space` (HF Space, always running)  
**Compute:** This Colab notebook (T4 GPU)  
**Goal:** Verify the full GRPO pipeline works end-to-end in 1 epoch

```
Colab T4 GPU  ──(HTTP)──►  HF Space Environment
(model lives here)         (scorer lives here)
```

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — switch runtime to T4')
print('VRAM:', f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB" if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'Go to Runtime > Change runtime type > T4 GPU'

In [ ]:
# Install dependencies (~3 minutes)
!pip install -q unsloth
!pip install -q --upgrade trl accelerate peft
!pip install -q requests
print('Done')

In [ ]:
import requests

ENV_URL = 'https://pathikreet-ap-clerk-env.hf.space'

# Verify the environment is reachable
health = requests.get(f'{ENV_URL}/health', timeout=30).json()
print('Environment status:', health['status'])
print('Version:', health.get('version'))
print('Total tasks:', health.get('total_tasks'))

tasks = requests.get(f'{ENV_URL}/tasks', timeout=30).json()
print(f'Tasks available: {len(tasks)}')
for t in tasks[:5]:
    print(f"  {t['task_id']} ({t['difficulty']})")
print('  ...')

In [ ]:
# Quick sanity check: one full episode manually
reset = requests.post(f'{ENV_URL}/reset',
                      json={'task_id': 'easy_perfect_match', 'seed': 42}).json()
session_id = reset['session_id']
obs = reset['observation']
print(f"Task: {obs['task_name']}")
print(f"Invoice total: ${obs['invoice']['invoice_total']:,.2f}")
print(f"Vendor: {obs['invoice']['vendor_name']}")

step = requests.post(f'{ENV_URL}/step', json={
    'session_id': session_id,
    'action': {
        'decision': 'APPROVE_FULL',
        'approved_amount': obs['invoice']['invoice_total'],
        'reason_code': 'MATCH_CONFIRMED',
        'explanation': f"Invoice matches PO and GRN. Total ${obs['invoice']['invoice_total']:.2f} approved."
    }
}).json()

print(f"Score: {step['reward']['score']}")
print(f"Feedback: {step['reward']['feedback']}")
print('Environment working correctly!')

In [ ]:
# Load model — using Qwen2.5-1.5B for fast pipeline test
# (swap to Meta-Llama-3-8B-Instruct-bnb-4bit for actual training)
from unsloth import FastLanguageModel

MODEL_NAME = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'  # tiny model, ~1GB, fast
# MODEL_NAME = 'unsloth/Meta-Llama-3-8B-Instruct-bnb-4bit'  # use this for real training

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=8,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

print(f'Model loaded: {MODEL_NAME}')
model.print_trainable_parameters()

In [ ]:
import json, re, random

SYSTEM_PROMPT = """You are an AI Accounts Payable Clerk. Review the invoice, PO, and GRN, then output ONLY valid JSON:
{"decision": "APPROVE_FULL"|"APPROVE_PARTIAL"|"REJECT"|"ESCALATE"|"QUERY_VENDOR",
 "approved_amount": <float>,
 "reason_code": "MATCH_CONFIRMED"|"QUANTITY_MISMATCH"|"PRICE_DISCREPANCY"|"POLICY_VIOLATION"|"NO_PO_FOUND"|"DUPLICATE_INVOICE"|"VENDOR_MISMATCH"|"TAX_DISCREPANCY"|"PENDING_CLARIFICATION"|"MANAGER_REVIEW",
 "explanation": "<cite specific $ amounts>"}"""


def obs_to_prompt(obs: dict) -> str:
    inv = obs['invoice']
    lines = '\n'.join(
        f"  {li['description']}: qty={li['quantity']}, unit_price=${li['unit_price']:.2f}"
        for li in inv.get('line_items', [])
    )
    pos = '\n'.join(
        f"  PO {p['po_number']} ({p['status']}) {p['vendor_name']}: " +
        ', '.join(f"{l['description']} qty={l['ordered_quantity']} @${l['agreed_unit_price']:.2f}"
                  for l in p.get('lines', []))
        for p in obs.get('purchase_orders', [])
    )
    grns = '\n'.join(
        f"  GRN {g['grn_id']} (PO {g['po_number']}): " +
        ', '.join(f"{l['description']} recv={l['received_quantity']}"
                  for l in g.get('lines', []))
        for g in obs.get('goods_receipts', [])
    )
    context = '\n'.join(f'  {n}' for n in obs.get('context_notes', []))
    paid = ', '.join(obs.get('paid_invoice_ids', []))
    return (
        f"TASK: {obs['task_name']}\n{obs['task_description']}\n\n"
        f"INVOICE {inv['invoice_id']} | {inv['vendor_name']} | ${inv['invoice_total']:,.2f}\n{lines}\n"
        f"Freight: ${inv.get('freight_charge',0):.2f}\n\n"
        f"PURCHASE ORDERS:\n{pos}\n\nGOODS RECEIPTS:\n{grns}\n"
        + (f"PAID LEDGER: {paid}\n" if paid else "")
        + (f"CONTEXT:\n{context}\n" if context else "")
        + f"\nPOLICY:\n{obs['company_policy']}\n\nOutput JSON decision."
    )


def parse_action(raw: str) -> dict:
    clean = re.sub(r'```(?:json)?\s*|\s*```', '', raw).strip()
    m = re.search(r'\{.*\}', clean, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except Exception:
            pass
    return {'decision': 'REJECT', 'approved_amount': 0.0,
            'reason_code': 'NO_PO_FOUND', 'explanation': 'parse error fallback'}


def run_episode(task_id: str, action_json: dict, seed: int = None) -> float:
    """Reset env, submit one action, return score."""
    try:
        r = requests.post(f'{ENV_URL}/reset',
                          json={'task_id': task_id, 'seed': seed}, timeout=15)
        data = r.json()
        session_id = data['session_id']
        obs = data['observation']

        # If multi-step, check if action is intermediate
        step_r = requests.post(f'{ENV_URL}/step',
                               json={'session_id': session_id, 'action': action_json},
                               timeout=15).json()
        return float(step_r['reward']['score'])
    except Exception as e:
        print(f'  env error: {e}')
        return 0.01


print('Helper functions ready')

In [ ]:
from datasets import Dataset

# Tasks for 1-epoch pipeline test (keep small)
TRAIN_TASKS = [
    'easy_perfect_match',
    'easy_no_po_found',
    'medium_quantity_shortfall',
    'medium_price_discrepancy',
    'hard_policy_violation',
    'hard_duplicate_invoice',
]

# Build prompts dataset
rows = []
for task_id in TRAIN_TASKS:
    for seed in [1, 2, 3]:  # 3 seeds per task = 18 samples total
        try:
            reset = requests.post(f'{ENV_URL}/reset',
                                  json={'task_id': task_id, 'seed': seed},
                                  timeout=15).json()
            obs = reset['observation']
            prompt_text = obs_to_prompt(obs)
            messages = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': prompt_text},
            ]
            rows.append({
                'prompt': tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                ),
                'task_id': task_id,
                'seed':    seed,
            })
        except Exception as e:
            print(f'  skip {task_id} seed={seed}: {e}')

dataset = Dataset.from_list(rows)
print(f'Dataset: {len(dataset)} samples')
print('Sample prompt (first 300 chars):')
print(dataset[0]['prompt'][:300])

In [ ]:
# Baseline scores BEFORE training
FastLanguageModel.for_inference(model)

EVAL_TASKS = ['easy_perfect_match', 'easy_no_po_found',
              'medium_quantity_shortfall', 'hard_policy_violation']

def eval_task(task_id: str, seed: int = 99) -> float:
    reset = requests.post(f'{ENV_URL}/reset',
                          json={'task_id': task_id, 'seed': seed}, timeout=15).json()
    obs = reset['observation']
    session_id = reset['session_id']

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': obs_to_prompt(obs)},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, temperature=0.1, do_sample=True)
    raw = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    action = parse_action(raw)

    score = float(requests.post(f'{ENV_URL}/step',
                                json={'session_id': session_id, 'action': action},
                                timeout=15).json()['reward']['score'])
    return score, raw[:150]

print('=== BASELINE (before training) ===')
baseline = {}
for t in EVAL_TASKS:
    score, raw = eval_task(t)
    baseline[t] = score
    print(f'  {t}: {score:.3f}  |  {raw[:80]}')
print(f'  Mean: {sum(baseline.values())/len(baseline):.3f}')

In [ ]:
# GRPO reward function — calls HF Space for each completion
import time

def reward_fn(completions, prompts=None, task_id=None, seed=None, **kwargs):
    rewards = []
    task_ids = task_id if task_id is not None else ['easy_perfect_match'] * len(completions)
    seeds    = seed    if seed    is not None else [random.randint(1,999)] * len(completions)

    for completion, tid, s in zip(completions, task_ids, seeds):
        action = parse_action(completion)
        score  = run_episode(tid, action, seed=int(s))
        rewards.append(score)

    return rewards

# Quick smoke test of reward function
test_scores = reward_fn(
    ['{"decision": "APPROVE_FULL", "approved_amount": 100.0, "reason_code": "MATCH_CONFIRMED", "explanation": "Invoice $100 matches PO and GRN."}'],
    task_id=['easy_perfect_match'],
    seed=[42]
)
print(f'Reward function smoke test score: {test_scores[0]:.3f}')
print('Reward function working!')

In [ ]:
from trl import GRPOConfig, GRPOTrainer
FastLanguageModel.for_training(model)

config = GRPOConfig(
    output_dir            = './ap_commander_grpo',
    num_train_epochs      = 1,          # 1 epoch — pipeline test
    per_device_train_batch_size = 2,    # small batch for T4
    num_generations       = 4,          # 4 completions per prompt (GRPO group)
    learning_rate         = 2e-5,
    max_completion_length = 250,        # short JSON outputs
    temperature           = 0.9,
    logging_steps         = 1,          # log every step so we can see progress
    save_steps            = 999,        # don't save mid-run
    report_to             = 'none',
    remove_unused_columns = False,      # keep task_id and seed in batch
)

trainer = GRPOTrainer(
    model             = model,
    processing_class  = tokenizer,
    reward_funcs      = reward_fn,
    args              = config,
    train_dataset     = dataset,
)

print(f'Training on {len(dataset)} samples, 1 epoch')
print(f'Expected steps: {len(dataset) // config.per_device_train_batch_size}')
print('Starting...')

result = trainer.train()
print('Training complete!')
print(f'Final training loss: {result.training_loss:.4f}')

In [ ]:
# Scores AFTER training
FastLanguageModel.for_inference(model)

print('=== POST-TRAINING ===')
post = {}
for t in EVAL_TASKS:
    score, raw = eval_task(t)
    post[t] = score
    print(f'  {t}: {score:.3f}  |  {raw[:80]}')
print(f'  Mean: {sum(post.values())/len(post):.3f}')

print('\n=== COMPARISON ===')
for t in EVAL_TASKS:
    delta = post[t] - baseline[t]
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '=')
    print(f'  {t:<35} {baseline[t]:.3f} → {post[t]:.3f}  {arrow} {delta:+.3f}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

tasks  = list(EVAL_TASKS)
before = [baseline[t] for t in tasks]
after  = [post[t]     for t in tasks]
x      = np.arange(len(tasks))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - 0.2, before, 0.4, label='Before (1-epoch GRPO)', color='#e74c3c', alpha=0.8)
ax.bar(x + 0.2, after,  0.4, label='After',                 color='#2ecc71', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([t.replace('_', '\n') for t in tasks], fontsize=9)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score (0.01 – 0.99)')
ax.set_title('AP Commander — GRPO Pipeline Test (1 Epoch)')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.4, label='0.5 baseline')
ax.legend()
plt.tight_layout()
plt.savefig('pipeline_test_results.png', dpi=120)
plt.show()
print('Saved: pipeline_test_results.png')

## What Just Ran

**1 epoch** = one pass through all 18 training samples with GRPO.

This was a **pipeline verification** — confirming that:
- Colab ↔ HF Space HTTP connection works
- Model generates valid JSON actions
- Reward function scores them correctly
- LoRA gradients backpropagate without errors
- Before/after scores are measurable

**For the onsite (April 25–26):** swap `MODEL_NAME` to `unsloth/Meta-Llama-3-8B-Instruct-bnb-4bit`, increase `num_train_epochs` to 3, `num_generations` to 8, and add all 20 tasks. That's the full training run.